# Forward feature engineering

This is the dedicated FWD research branch of `fantasy_premier_league.ipynb`. The root owns shared data construction and benchmark definitions; this notebook owns forward-specific features, diagnostics, model comparisons, and prospective challengers.

The first extracted benchmark emphasizes availability, recent xG and goals, share of team attacking output, and the opponent's venue-specific defence. Results here do not change the root model automatically.

In [1]:
import pandas as pd
from IPython.display import display

from position_research import compare_models

POSITION = "FWD"
K = 10
PANEL_PATH = "fpl_feature_panel.pkl"

panel = pd.read_pickle(PANEL_PATH)
position_panel = panel.loc[panel["position"].eq(POSITION)].copy()
print(POSITION, position_panel.shape, "K =", K)
display(position_panel.groupby("season")["GW"].agg(["min", "max", "nunique"]))

FWD (7074, 95) K = 10


,min,max,nunique
season,,,
2024-25,1,38,38
2025-26,1,38,38
2026-27,1,11,11


## Extracted FWD feature set

This is the current root challenger moved into its own research branch. Future forward work should test finishing-volume spikes, penalty responsibility, opponent chances conceded centrally, and home/away effects here before promotion.

In [2]:
forward_features = [
    "minutes_ewm", "zero_min_rate_5", "start_rate_5", "minutes_60_rate",
    "minutes_form", "xg_per90_5", "xa_per90_5", "xg_per90_10",
    "goals_per90_10", "assists_per90_10", "threat_per90_5",
    "xgi_team_rate_ratio_5", "bps_per90_5", "bonus_rate_5",
    "team_xg_venue_5", "opp_xga_venue_5", "venue_attack_matchup",
    "n_fixtures", "n_home", "value",
]

missing = sorted(set(forward_features) - set(position_panel.columns))
assert not missing, f"Missing root features: {missing}"
print("FWD features:", len(forward_features))
display(pd.Series(forward_features, name="feature").to_frame())

FWD features: 20


,feature
0,minutes_ewm
1,zero_min_rate_5
2,start_rate_5
3,minutes_60_rate
4,minutes_form
5,xg_per90_5
6,xa_per90_5
7,xg_per90_10
8,goals_per90_10
9,assists_per90_10


## Leak-safe walk-forward benchmark

Each evaluated gameweek trains only on earlier rows. The target and predicted shortlist are tie-aware top K, and each training gameweek receives equal total weight. GW1-19 is development; GW20-38 is a historical check.

In [3]:
forward_summary, forward_detail = compare_models(
    position_panel,
    forward_features,
    K,
)
display(forward_summary)
display(
    forward_summary.pivot(index="model", columns="block", values="mean_hit_rate")
)

,block,model,mean_hit_rate,correct,predictions,weeks
0,development_gw1_19,gradient_boosting,0.600000,114,190,19
1,development_gw1_19,logistic_regression,0.600000,114,190,19
2,development_gw1_19,minutes_sort,0.563158,107,190,19
4,historical_check_gw20_38,logistic_regression,0.505263,96,190,19
5,historical_check_gw20_38,minutes_sort,0.505263,96,190,19
3,historical_check_gw20_38,gradient_boosting,0.473684,90,190,19


block,development_gw1_19,historical_check_gw20_38
model,,
gradient_boosting,0.600000,0.473684
logistic_regression,0.600000,0.505263
minutes_sort,0.563158,0.505263
